In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print("GROQ_API_KEY configured:", bool(groq_api_key))

In [1]:
from langchain_groq import ChatGroq

# Replace this model if your Groq dashboard shows a different available model.
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

In [5]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

PDF_DIR = Path(r"P:\ADVANCED_RAG\Hybrid_Search_RAG\documents")

raw_docs = []
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print("=" * 70)
print("PDF DOCUMENT LOADING")
print("=" * 70)

print(f"Folder       : {PDF_DIR}")
print(f"PDF files    : {len(pdf_files)}")
print()


for pdf_path in pdf_files:
    print(f"Loading: {pdf_path.name}")

    try:
        loader = PyPDFLoader(str(pdf_path))
        docs = loader.load()
        # Add useful metadata
        for doc in docs:
            doc.metadata["file_name"] = pdf_path.name
            doc.metadata["file_path"] = str(pdf_path)
        raw_docs.extend(docs)
        print(f"  Pages loaded: {len(docs)}")
    except Exception as e:
        print(f"  ERROR: {e}")



print("LOADING COMPLETE")
print(f"PDF files loaded : {len(pdf_files)}")
print(f"Total pages      : {len(raw_docs)}")

Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 64 0 (offset 0)
Ignoring wrong pointing object 137 0 (offset 0)
Ignoring wrong pointing object 177 0 (offset 0)
Ignoring wrong pointing object 179 0 (offset 0)
Ignoring wrong pointing object 181 0 (offset 0)
Ignoring wrong pointing object 202 0 (offset 0)
Ignoring wrong pointing object 287 0 (offset 0)
Ignoring wrong pointing object 289 0 (offset 0)
Ignoring wrong pointing object 311 0 (offset 0)
Ignoring wrong pointing object 313 0 (offset 0)
Ignoring wrong pointing object 315 0 (offset 0)
Ignoring wrong pointing object 391 0 (offset 0)
Ignoring wrong pointing object 412 0 (offset 0)


PDF DOCUMENT LOADING
Folder       : P:\ADVANCED_RAG\Hybrid_Search_RAG\documents
PDF files    : 7

Loading: covid-19.pdf
  Pages loaded: 18
Loading: Dengue.pdf
  Pages loaded: 9
Loading: depression.pdf
  Pages loaded: 8
Loading: diabetes.pdf
  Pages loaded: 23
Loading: hepatitis C.pdf
  Pages loaded: 3
Loading: Hepatitis-B-Fast-Facts.pdf
  Pages loaded: 2
Loading: Malaria.pdf
  Pages loaded: 23
LOADING COMPLETE
PDF files loaded : 7
Total pages      : 86


In [8]:
for i, doc in enumerate(raw_docs[:10], 1):
    print("Document:", i)
    print("File:", doc.metadata.get("file_name"))
    print("Page:", doc.metadata.get("page", 0) + 1)

    print("\nContent:")
    print(doc.page_content[:100])

Document: 1
File: covid-19.pdf
Page: 1

Content:
The picture can't be 
displayed.
The picture can't 
be displayed.
What is COVID-19
•COVID-19 is the 
Document: 2
File: covid-19.pdf
Page: 2

Content:
The picture can't be 
displayed.
The picture can't 
be displayed.
Immunity Personal 
measures 
Disea
Document: 3
File: covid-19.pdf
Page: 3

Content:
The picture can't be 
displayed.
The picture can't 
be displayed.
Origin of the virus
The source is 
Document: 4
File: covid-19.pdf
Page: 4

Content:
The picture can't be 
displayed.
The picture can't 
be displayed.
Symptoms 
Symptoms: 1/1
Document: 5
File: covid-19.pdf
Page: 5

Content:
The picture can't be 
displayed.
The picture can't 
be displayed.
Transmission 
Transmission: 1/2
CO
Document: 6
File: covid-19.pdf
Page: 6

Content:
The picture can't be 
displayed.
The picture can't 
be displayed.
Transmission
Transmission: 2/2
Act
Document: 7
File: covid-19.pdf
Page: 7

Content:
The picture can't be 
displayed.
The picture can't 
be display

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = splitter.split_documents(raw_docs)

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:900])

Total chunks: 224

First chunk preview:

The picture can't be 
displayed.
The picture can't 
be displayed.
What is COVID-19
•COVID-19 is the infectious disease caused by the most recently 
discovered coronavirus1
•This new virus and disease were unknown before the outbreak 
began in Wuhan, China, in December 2019
•COVID-19 is now a pandemic affecting many countries globally
1https://www.who.int/emergencies/diseases/novel-coronavirus-2019/question-and-answers-hub/q-a-detail/q-a-coronaviruses


In [10]:
chunks

[Document(metadata={'producer': 'macOS Version 10.15.5 (Build 19F101) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20200623103112Z00'00'", 'moddate': "D:20200623103112Z00'00'", 'source': 'P:\\ADVANCED_RAG\\Hybrid_Search_RAG\\documents\\covid-19.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'file_name': 'covid-19.pdf', 'file_path': 'P:\\ADVANCED_RAG\\Hybrid_Search_RAG\\documents\\covid-19.pdf', 'start_index': 0}, page_content="The picture can't be \ndisplayed.\nThe picture can't \nbe displayed.\nWhat is COVID-19\n•COVID-19 is the infectious disease caused by the most recently \ndiscovered coronavirus1\n•This new virus and disease were unknown before the outbreak \nbegan in Wuhan, China, in December 2019\n•COVID-19 is now a pandemic affecting many countries globally\n1https://www.who.int/emergencies/diseases/novel-coronavirus-2019/question-and-answers-hub/q-a-detail/q-a-coronaviruses"),
 Document(metadata={'producer': 'macOS Version 10.15.5 (Build 19F101) Quartz PD

In [11]:
from langchain_community.retrievers import BM25Retriever

# BM25 should use the SAME chunks that Chroma uses
bm25_retriever = BM25Retriever.from_documents(
    chunks
)

bm25_retriever.k = 4

print("BM25 retriever initialized")

BM25 retriever initialized


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

sample_vector = embeddings.embed_query("What is Agentic RAG?")
print("Embedding dimensions:", len(sample_vector))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2922.07it/s]


Embedding dimensions: 384


In [13]:
from langchain_chroma import Chroma

PERSIST_DIR = "./pdf_hybrid_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="pdf_demo",
)

print("Vectorstore created and persisted at:", PERSIST_DIR)

# Sanity check
query = "What is Agentic RAG?"
results = vectorstore.similarity_search(query, k=3)

print(f"\nTop {len(results)} results for query: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:300])
    print()

Vectorstore created and persisted at: ./pdf_hybrid_db

Top 3 results for query: 'What is Agentic RAG?'

--- Result 1 ---
Page: 5
sleeping areas should be sprayed before bedtime. Treating a room with an insecticide spray will 
help to free it from insects, but the effect may be short-lived. Spraying before bedtime, combined 
with the use of a vaporizer or a mosquito net, is r ecommended. Aerosol sprays intended for 
crawling i

--- Result 2 ---
Page: 1
during pregnancy or childbirth 
Sharing razors, toothbrushes, nail clippers, 
earrings or body jewelry with an infected 
person 
Human bites from an infected person 
Unsterilized instruments used by roadside 
dentists, doctors, and barbers 
Kissing on the cheek or lips 
Coughing or sneezing 
Casual 

--- Result 3 ---
Page: 6
D E N V  i n f e c t i o n  c a n  b e  d i a g n o s e d  u s i n g  a  v a r i e t y  o f  
m e t h o d s .  T h e s e  i n c l u d e  t h e  v i r o l o g i c a l  t e s t s  ( w h i c h  
detect virus components d

In [15]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [16]:
from langchain_classic.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        retriever
    ],
    weights=[
        0.5,  # BM25
        0.5   # Chroma
    ]
)

print("Hybrid retriever initialized")

Hybrid retriever initialized


In [17]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu"
)

print("Reranker loaded!")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2354.59it/s]


Reranker loaded!


In [18]:
# =========================================================
# 8. RERANK DOCUMENTS
# =========================================================

def rerank_documents(
    query,
    documents,
    top_k=3
):

    if not documents:
        return []

    pairs = [
        [query, doc.page_content]
        for doc in documents
    ]

    scores = reranker.predict(pairs)

    ranked_documents = sorted(
        zip(scores, documents),
        key=lambda x: x[0],
        reverse=True
    )

    final_documents = []

    for score, doc in ranked_documents[:top_k]:

        doc.metadata["rerank_score"] = float(score)

        final_documents.append(doc)

    return final_documents

In [19]:
# =========================================================
# 9. HYBRID SEARCH + RERANKING
# =========================================================

def retrieve_documents(
    query,
    final_k=3
):

    # -----------------------------------------
    # STEP 1: Hybrid retrieval
    # -----------------------------------------

    retrieved_docs = ensemble_retriever.invoke(
        query
    )

    print(
        f"Hybrid retrieval returned: "
        f"{len(retrieved_docs)} documents"
    )

    # -----------------------------------------
    # STEP 2: Reranking
    # -----------------------------------------

    reranked_docs = rerank_documents(
        query=query,
        documents=retrieved_docs,
        top_k=final_k
    )

    return reranked_docs

In [20]:
query = "Which invoice has GSTIN 19ABCDE1234F1Z5?"

final_docs = retrieve_documents(
    query,
    final_k=3
)

Hybrid retrieval returned: 14 documents


In [21]:
for rank, doc in enumerate(final_docs, 1):
    
    print("=" * 80)

    print(f"RANK: {rank}")

    print(
        "Rerank Score:",
        doc.metadata.get("rerank_score")
    )

    print(
        "File:",
        doc.metadata.get("file_name")
    )

    print(
        "Page:",
        doc.metadata.get("page_number")
    )

    print("\nContent:")
    print(doc.page_content[:1000])

RANK: 1
Rerank Score: -10.319509506225586
File: covid-19.pdf
Page: None

Content:
The picture can't be 
displayed.
The picture can't 
be displayed.
Treatment
Currently, there are no antiviral drugs licensed for treating COVID-19. 
Research is ongoing to determine if existing drugs can be re-purposed to 
effectively treat COVID-19
WHO is coordinating the large multi-country Solidarity Trialto evaluate four 
promising candidate drugs/regimens:
Source:ID card
Treatment: 1/1
RANK: 2
Rerank Score: -10.363893508911133
File: covid-19.pdf
Page: None

Content:
The picture can't be 
displayed.
The picture can't 
be displayed.
What is COVID-19
•COVID-19 is the infectious disease caused by the most recently 
discovered coronavirus1
•This new virus and disease were unknown before the outbreak 
began in Wuhan, China, in December 2019
•COVID-19 is now a pandemic affecting many countries globally
1https://www.who.int/emergencies/diseases/novel-coronavirus-2019/question-and-answers-hub/q-a-detail/q-a-c

In [22]:
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template(
    """
You are a document question-answering assistant.

Answer the question using ONLY the provided document context.

Do not use outside knowledge.

If the answer cannot be found in the provided documents,
say:

"I don't have enough information in the provided documents."

Always mention the source file and page number when available.

CONTEXT:

{context}

QUESTION:

{question}

ANSWER:
"""
)

In [24]:
def format_docs(docs):
    
    formatted_documents = []

    for doc in docs:

        formatted_documents.append(
            f"""
SOURCE FILE: {doc.metadata.get("file_name")}
PAGE: {doc.metadata.get("page_number")}

CONTENT:
{doc.page_content}
"""
        )

    return "\n\n".join(
        formatted_documents
    )

In [25]:
from langchain_core.output_parsers import StrOutputParser


def ask_question(
    question,
    final_k=3
):

    # =========================================
    # STEP 1: Hybrid + Reranking
    # =========================================

    final_docs = retrieve_documents(
        query=question,
        final_k=final_k
    )

    # =========================================
    # STEP 2: Format context
    # =========================================

    context = format_docs(
        final_docs
    )

    # =========================================
    # STEP 3: LLM
    # =========================================

    chain = (
        prompt
        | llm
        | StrOutputParser()
    )

    answer = chain.invoke({
        "context": context,
        "question": question
    })

    return {
        "question": question,
        "answer": answer,
        "documents": final_docs
    }

In [26]:
result = ask_question(
    "What are the test done in covid -19 ?"
)

print("\nQUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

Hybrid retrieval returned: 14 documents

QUESTION:
What are the test done in covid -19 ?

ANSWER:
The documents describe three main types of tests used for COVID‑19:

1. **Nucleic Acid Amplification Tests (NAATs)** – used to diagnose current infection by detecting the virus itself.  
2. **Antibody detection tests** – detect IgG and IgM antibodies against the virus; useful for identifying past infection but not for early diagnosis.  
3. **Antigen detection tests** – detect viral antigens and can be used for diagnosing acute infection.

These tests are currently available only under emergency‑use regulations.  

*Source: covid‑19.pdf (pages not specified).*


In [27]:
print("\nSOURCES:")
print("=" * 80)

for i, doc in enumerate(
    result["documents"],
    1
):

    print(f"\nSource {i}")
    print(
        "File:",
        doc.metadata.get("file_name")
    )
    print(
        "Page:",
        doc.metadata.get("page_number")
    )
    print(
        "Rerank Score:",
        doc.metadata.get("rerank_score")
    )


SOURCES:

Source 1
File: covid-19.pdf
Page: None
Rerank Score: 2.2461652755737305

Source 2
File: covid-19.pdf
Page: None
Rerank Score: 0.47255316376686096

Source 3
File: covid-19.pdf
Page: None
Rerank Score: -0.2837732136249542
